**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/grosmark-2016"], check=True)
    os.chdir("/content/encore/entries/grosmark-2016")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings on DANDI:000044, version 0.261106.0236

This notebook belongs to the reconciliation of attempt grosmark-2016-02 with the attempt before it, grosmark-2016-01, which was made on version 0.261005.2101 of DANDI:000044. Each section below takes one of the earlier attempt's findings that this attempt did not report, opens with the earlier statement, and checks it on version 0.261106.0236 (`g.DANDI_VERSION`). The earlier finding 2 (the end of the POST epoch of Gatsby_08022013) is reported by this attempt's own finding 2, and is not repeated here.

The NWB files are streamed with remfile and h5py, as in `reproduce_grosmark_2016.ipynb`; only metadata, the position series and table columns are read, not the LFP.

In [ ]:
import re
import numpy as np
import pandas as pd
import requests
import grosmark as g

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
print("DANDI", g.DANDISET, "version", g.DANDI_VERSION)

In [ ]:
# sanity: the files of the version, and what each holds at its top level
ASSETS = g.dandi_assets()
F = {}
rows = []
for a in ASSETS:
    name = a["path"].split("ses-")[1].split("_")[0].replace("-", "_")
    f = g.open_nwb(a["asset_id"])
    F[name] = f
    ep = f["intervals/epochs"]
    lab = list(g._dec(ep["label"][:]))
    st, sp = ep["start_time"][:], ep["stop_time"][:]
    rows.append(dict(session=name, path=a["path"], size_GB=round(a["size"] / 1e9, 2), epochs=",".join(lab),
                     MAZE_s=round(float(sp[lab.index("MAZE")] - st[lab.index("MAZE")]), 1),
                     acquisition=",".join(f["acquisition"].keys()), processing=",".join(f["processing"].keys()),
                     intervals=",".join(f["intervals"].keys())))
FILES = pd.DataFrame(rows).set_index("session")
print(len(FILES), "files")
print(FILES.drop(columns="path").to_string())

## Earlier finding 1

> In versions 0.210812.1516 and 0.250624.0426 every position series stores its sampling period (0.0256 s) as its rate, so the positions span about 880 to 2360 hours in place of the MAZE epoch. Version 0.261005.2101 stores 39.06 Hz, which matches the epoch lengths.

The rate stored with each position series, and the time its samples span at that rate, against the length of the MAZE epoch.

In [ ]:
rows = []
for name, f in F.items():
    for key in ("linearized_position", "position_2d"):
        ser = f[f"processing/behavior/position/{key}"]
        n = ser["data"].shape[0]
        if "timestamps" in ser:
            ts = ser["timestamps"]
            span, rate = float(ts[-1] - ts[0]), np.nan
        else:
            rate = float(ser["starting_time"].attrs["rate"])
            span = n / rate
        rows.append(dict(session=name, series=key, samples=n, rate_Hz=round(rate, 4), span_s=round(span, 1),
                         MAZE_s=FILES.loc[name, "MAZE_s"], span_over_MAZE=round(span / FILES.loc[name, "MAZE_s"], 3)))
POS = pd.DataFrame(rows)
print(POS.to_string(index=False))
print("rates:", POS.rate_Hz.min(), "to", POS.rate_Hz.max(), "Hz; span over MAZE:", POS.span_over_MAZE.min(), "to", POS.span_over_MAZE.max())

## Earlier finding 3

> The dandiset description lists neck EMG and head-mounted accelerometer signals, and the files hold neither; they also hold no wide-band data (acquisition is empty), only the LFP at 1250 Hz.

The series under `acquisition` in each file, with their shapes and rates, and the start of their descriptions.

In [ ]:
desc = requests.get(f"https://api.dandiarchive.org/api/dandisets/{g.DANDISET}/versions/{g.DANDI_VERSION}/info/",
                    timeout=60).json()["metadata"]["description"]
print("dandiset description, the sentence on other signals:",
      [s for s in desc.split(". ") if "EMG" in s or "accelerometer" in s])
rows = []
for name, f in F.items():
    for key, ser in f["acquisition"].items():
        rows.append(dict(session=name, series=key, shape=ser["data"].shape,
                         rate_Hz=float(ser["starting_time"].attrs["rate"]) if "starting_time" in ser else np.nan))
print(pd.DataFrame(rows).to_string(index=False))
f = next(iter(F.values()))
for key, ser in f["acquisition"].items():
    print(key, ":", ser.attrs["description"][:300])
print("LFP:", f["processing/ecephys/LFP/lfp"].attrs["description"][:200])

## Earlier finding 4

> The paper's derived data are not deposited. There are no candidate or ripple events, no place-cell list, no sequence scores and no PCC values or rigid and plastic labels, so the paper's selections cannot be checked against the dandiset and every one must be rebuilt.

Every group and table under `processing`, `analysis` and `intervals` in each file, the columns of the units table, and any name that refers to events, ripples, place cells, sequence scores, the PCC or the rigid and plastic classes.

In [ ]:
PAT = re.compile(r"ripple|event|candidate|place|replay|sequence|score|pcc|rigid|plastic", re.I)
rows = []
for name, f in F.items():
    objs = []
    for top in ("processing", "analysis", "intervals"):
        f[top].visit(lambda p, top=top: objs.append(f"{top}/{p}"))
    containers = [p for p in objs if "meanings_tables" not in p and p.count("/") <= 2]
    unit_cols = list(f["units"].keys())
    hits = [p for p in objs + [f"units/{c}" for c in unit_cols] if PAT.search(p.split("/")[-1])]
    rows.append(dict(session=name, groups=", ".join(containers), units_columns=", ".join(unit_cols),
                     analysis_items=len(f["analysis"]), matching_names=", ".join(hits) or "none"))
DER = pd.DataFrame(rows).set_index("session")
print(DER[["analysis_items", "matching_names"]].to_string())
print()
print("groups of the first file:", DER.groups.iloc[0])
print("units columns of the first file:", DER.units_columns.iloc[0])

## Earlier finding 5

> Neither the paper nor the dandiset links analysis code, and the supplementary Materials and Methods, which define the candidate events, place cells, the PCC and its test, are not openly available (PMC and Science serve them only behind a browser check).

The dandiset's related resources on this version, and the code that each NWB file names. Whether the supplement could be read is not a property of the files: the claims file of this attempt (`claims/grosmark-2016.yaml`) was written from the PubMed Central author manuscript with its supplement, so the Materials and Methods were available to it.

In [ ]:
meta = requests.get(f"https://api.dandiarchive.org/api/dandisets/{g.DANDISET}/versions/{g.DANDI_VERSION}/info/",
                    timeout=60).json()["metadata"]
for r in meta.get("relatedResource", []):
    print("related:", r.get("relation"), r.get("resourceType"), r.get("url"))
print("URLs in the description:", re.findall(r"https?://\S+", meta["description"]) or "none")
print("protocol:", meta.get("protocol"))
for name, f in F.items():
    src = f["general/source_script"][()].decode() if "general/source_script" in f else ""
    print(name, "source_script repository:", re.findall(r"repository: (\S+)", src), "; related_publications:",
          [x.decode() for x in f["general/related_publications"][:]])

## Earlier finding 6

> Every electrode's location is "unknown" and no depth or layer is given, so the channel for ripple detection has to be chosen from the signals. Units carry only the hemisphere (lCA1 or rCA1). Subject age is "/".

The electrodes' locations (with the groups of those still "unknown"), the electrode groups' descriptions, the location columns of the units table, and the subject's age with its description.

In [ ]:
rows = []
for name, f in F.items():
    el = f["general/extracellular_ephys/electrodes"]
    loc = g._dec(el["location"][:])
    grp = g._dec(el["group_name"][:])
    unk = sorted(set(grp[loc == "unknown"]))
    groups = f["general/extracellular_ephys"]
    layer = sum("pyramidal layer" in groups[k].attrs.get("description", "") for k in groups if k != "electrodes")
    rows.append(dict(session=name, electrodes=len(loc), CA1=int((loc == "CA1 field of hippocampus").sum()),
                     unknown=int((loc == "unknown").sum()), unknown_groups=",".join(unk) or "-",
                     groups_naming_layer=layer, units=len(f["units/id"]),
                     unit_hemispheres=",".join(sorted(set(g._dec(f["units/hemisphere"][:])))),
                     age=f["general/subject/age"][()].decode()))
EL = pd.DataFrame(rows).set_index("session")
print(EL.to_string())
f = next(iter(F.values()))
groups = f["general/extracellular_ephys"]
for k in [k for k in groups if k != "electrodes"][:1] + [k for k in groups if k in set(EL.unknown_groups.str.split(",").sum())][:2]:
    print(k, ":", groups[k].attrs.get("description", ""))
print("subject description:", f["general/subject/description"][()].decode())

## Earlier finding 7

> On the circular track the animals ran toward decreasing linearized position (47 to 63% of samples against 7 to 16% the other way). The series description says the position increases clockwise but not which way the animals ran.

For the circular-track sessions: the description of the linearized position, and the fractions of samples moving toward lower and higher positions at more than 5 cm/s, from the stored samples and rate.

In [ ]:
rows = []
for name, f in F.items():
    ser = f["processing/behavior/position/linearized_position"]
    d = ser.attrs["description"]
    if "circular" not in d:
        continue
    x = ser["data"][:]
    rate = float(ser["starting_time"].attrs["rate"])
    v = np.diff(x) * rate
    ok = np.isfinite(v) & (np.abs(v) < 2.0)      # drop the jump where the position wraps around the track
    rows.append(dict(session=name, lower=round(float(np.mean(v[ok] < -0.05)), 3), higher=round(float(np.mean(v[ok] > 0.05)), 3),
                     states_direction="ran mostly toward decreasing linearized position" in d))
    print(name, ":", d[:520])
    print()
print(pd.DataFrame(rows).to_string(index=False))